In [1]:
import math
from time import perf_counter_ns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

from zoo.utils import get_device

In [2]:
dtype = torch.bfloat16
device = get_device()
print(f"Using dtype: {dtype}")
print(f"Using device: {device}")

Using dtype: torch.bfloat16
Using device: cuda


In [3]:
import re


class Tokenizer:
    def __init__(
        self, dataset: str, sos_token: str, eos_token: str, unknown_token: str
    ):
        # CHANGE: sorted() — set iteration order is non-deterministic across runs,
        # so vocab indices would change every time, making saved models unloadable.
        self.vocab = sorted(set(dataset))
        self.vocab.insert(0, sos_token)
        self.vocab.insert(1, eos_token)
        self.vocab.insert(2, unknown_token)
        self.word2idx = {ch: idx for idx, ch in enumerate(self.vocab)}
        self.idx2word = {idx: ch for idx, ch in enumerate(self.vocab)}

    def __len__(self):
        return len(self.vocab)

    def apply_chat_formatting(self, text: str) -> str:
        return f"<|startoftext|>{text}<|endoftext|>"

    def encode(self, text: str, add_special_tokens: bool = True) -> list[int]:
        if add_special_tokens:
            text = self.apply_chat_formatting(text)
        return [self.word2idx.get(ch, self.word2idx["<|unk|>"]) for ch in text]

    def encode_batch(
        self, texts: list[str], add_special_tokens: bool = True
    ) -> list[list[int]]:
        return [self.encode(text, add_special_tokens) for text in texts]

    def decode(self, indices: list[int], skip_special_tokens: bool = True) -> str:
        text = "".join(self.idx2word[idx] for idx in indices)
        if skip_special_tokens:
            text = re.sub(r"<\|.*?\|>", "", text)
        return text

    def decode_batch(
        self, batch_indices: list[list[int]], skip_special_tokens: bool = True
    ) -> list[str]:
        return [self.decode(indices, skip_special_tokens) for indices in batch_indices]


tokenizer = Tokenizer(
    " abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789!@#$%^&*()-_=+[]{}|;:,.<>?/~`\n",
    sos_token="<|startoftext|>",
    eos_token="<|endoftext|>",
    unknown_token="<|unk|>",
)
print(f"Vocab size: {len(tokenizer)}")

Vocab size: 96


In [4]:
# ── RoPE ──────────────────────────────────────────────────────────────────────
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_length=2048, base=10000):
        super().__init__()
        self.register_buffer(
            "inv_freq",
            1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim)),
            persistent=False,
        )
        self.max_seq_length = max_seq_length
        self._build_cache(max_seq_length)

    def _build_cache(self, seq_len):
        t = torch.arange(
            seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device
        )
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None], persistent=False)
        self.register_buffer("sin_cached", emb.sin()[None, None], persistent=False)

    def forward(self, seq_len: int):
        # Rebuild cache if device moved after init (e.g. .to(mps) after construction)
        if self.cos_cached.device != self.inv_freq.device:
            self._build_cache(self.max_seq_length)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]


def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, cos, sin):
    return (q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin)


# ── Attention ─────────────────────────────────────────────────────────────────
class Attention(nn.Module):
    """
    Unified MLA + GQA with absorbed inference path.

    Training:  decompress c_kv → K_nope, V per kv_head group, GQA expand, attend.
    Inference: absorb W_uk into Q, absorb W_uv into W_o, attend directly on c_kv.
               KV cache = (c_kv, k_rope) — never materialized into full heads.

    The absorbed path is mathematically identical but avoids decompression matmuls,
    turning the attention into MQA-like compute against a single shared latent head.
    """

    def __init__(
        self,
        embed_size,
        num_heads,
        num_kv_heads,
        qk_nope_dim,
        qk_rope_dim,
        kv_latent_rank,
        q_latent_rank,
        v_head_dim,
        max_seq_length,
    ):
        super().__init__()
        assert embed_size % num_heads == 0
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads or num_heads
        assert num_heads % self.num_kv_heads == 0
        self.kv_group_size = num_heads // self.num_kv_heads

        head_dim = embed_size // num_heads
        self.qk_nope_dim = qk_nope_dim or (head_dim * 3 // 4)
        self.qk_rope_dim = qk_rope_dim or (head_dim - self.qk_nope_dim)
        self.v_head_dim = v_head_dim or head_dim
        self.kv_latent_rank = kv_latent_rank or (embed_size // 2)
        self.q_latent_rank = q_latent_rank or (embed_size // 2)

        # ── Q path ──
        self.w_dq = nn.Linear(embed_size, self.q_latent_rank, bias=False)
        self.q_norm = RMSNorm(self.q_latent_rank)
        self.w_uq = nn.Linear(
            self.q_latent_rank, num_heads * self.qk_nope_dim, bias=False
        )
        self.w_qr = nn.Linear(
            self.q_latent_rank, num_heads * self.qk_rope_dim, bias=False
        )

        # ── KV path ──
        self.w_dkv = nn.Linear(
            embed_size, self.kv_latent_rank + self.qk_rope_dim, bias=False
        )
        self.kv_norm = RMSNorm(self.kv_latent_rank)
        self.w_uk = nn.Linear(
            self.kv_latent_rank, self.num_kv_heads * self.qk_nope_dim, bias=False
        )
        self.w_uv = nn.Linear(
            self.kv_latent_rank, self.num_kv_heads * self.v_head_dim, bias=False
        )

        # ── Output ──
        self.fc_out = nn.Linear(num_heads * self.v_head_dim, embed_size, bias=False)

        # ── RoPE ──
        self.rope = RotaryPositionalEmbedding(self.qk_rope_dim, max_seq_length)

        # ── Absorbed weights (built lazily on first inference call) ──
        self._absorbed_ready = False

    # ──────────────────────────────────────────────────────────────────────
    # Absorption: pre-fuse W_uk into Q and W_uv into W_o for inference
    # ──────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def _build_absorbed_weights(self):
        """
        Pre-compute:
          W_uq_absorbed = W_uq @ W_uk^T   per KV group, so Q projects directly to latent dim
          W_o_absorbed  = W_o  @ W_uv      per KV group, so output absorbs value decompression

        After absorption, attention keys and values are both just c_kv (the raw latent),
        making this MQA-like: one shared "head" of dim kv_latent_rank.
        """
        # W_uk: (num_kv_heads * qk_nope_dim, kv_latent_rank)
        # Reshape to (num_kv_heads, qk_nope_dim, kv_latent_rank)
        w_uk = self.w_uk.weight.view(
            self.num_kv_heads, self.qk_nope_dim, self.kv_latent_rank
        )

        # W_uq: (num_heads * qk_nope_dim, q_latent_rank)
        # Reshape to (num_heads, qk_nope_dim, q_latent_rank)
        w_uq = self.w_uq.weight.view(
            self.num_heads, self.qk_nope_dim, self.q_latent_rank
        )

        # For each query head, find its KV group and compute W_uq_head^T @ W_uk_group
        # = (q_latent_rank, qk_nope_dim) @ (qk_nope_dim, kv_latent_rank)
        # = (q_latent_rank, kv_latent_rank)  per head
        absorbed_pieces = []
        for h in range(self.num_heads):
            g = h // self.kv_group_size
            # w_uq[h]: (qk_nope_dim, q_latent_rank), w_uk[g]: (qk_nope_dim, kv_latent_rank)
            piece = w_uq[h].T @ w_uk[g]  # (q_latent_rank, kv_latent_rank)
            absorbed_pieces.append(piece)
        # Stack → (num_heads, q_latent_rank, kv_latent_rank)
        # Register as a linear: input r_q → output num_heads * kv_latent_rank
        w_uq_absorbed = torch.stack(absorbed_pieces, dim=0)  # (H, r_q, r_kv)
        self.register_buffer(
            "_w_uq_absorbed",
            w_uq_absorbed.reshape(
                self.num_heads * self.kv_latent_rank, self.q_latent_rank
            ),
            persistent=False,
        )

        # W_uv: (num_kv_heads * v_head_dim, kv_latent_rank)
        # W_o:  (embed_size, num_heads * v_head_dim)
        # For each head h in KV group g:
        #   W_o_slice[h] @ W_uv[g] = (embed_size_slice, v_head_dim) @ (v_head_dim, kv_latent_rank)
        #   = (embed_size_slice... no, let's think per-head)
        #
        # W_o has shape (embed_size, num_heads * v_head_dim).
        # Slice column-wise per head: W_o_h = W_o[:, h*v_head_dim : (h+1)*v_head_dim]
        # W_uv_g = W_uv.weight reshaped to (num_kv_heads, v_head_dim, kv_latent_rank)
        # W_o_absorbed_h = W_o_h @ W_uv_g[g]  → (embed_size, kv_latent_rank)
        w_uv = self.w_uv.weight.view(
            self.num_kv_heads, self.v_head_dim, self.kv_latent_rank
        )
        w_o = self.fc_out.weight  # (embed_size, num_heads * v_head_dim)

        o_absorbed_pieces = []
        for h in range(self.num_heads):
            g = h // self.kv_group_size
            w_o_h = w_o[
                :, h * self.v_head_dim : (h + 1) * self.v_head_dim
            ]  # (E, v_head_dim)
            piece = w_o_h @ w_uv[g]  # (E, kv_latent_rank)
            o_absorbed_pieces.append(piece)
        # Stack → (num_heads, embed_size, kv_latent_rank)
        self.register_buffer(
            "_w_o_absorbed",
            torch.stack(o_absorbed_pieces, dim=0),  # (H, E, r_kv)
            persistent=False,
        )

        self._absorbed_ready = True

    def train(self, mode=True):
        if mode:
            self._absorbed_ready = False
        return super().train(mode)

    # ──────────────────────────────────────────────────────────────────────
    # Training forward: full decompress + GQA expand
    # ──────────────────────────────────────────────────────────────────────
    def _forward_train(self, x, kv_cache=None, use_cache=False):
        B, S, E = x.shape

        # Q
        c_q = self.q_norm(self.w_dq(x))
        q_nope = (
            self.w_uq(c_q).view(B, S, self.num_heads, self.qk_nope_dim).transpose(1, 2)
        )
        q_rope = (
            self.w_qr(c_q).view(B, S, self.num_heads, self.qk_rope_dim).transpose(1, 2)
        )

        # KV
        kv_out = self.w_dkv(x)
        c_kv, k_rope = torch.split(
            kv_out, [self.kv_latent_rank, self.qk_rope_dim], dim=-1
        )
        c_kv = self.kv_norm(c_kv)
        k_rope = k_rope.view(B, S, 1, self.qk_rope_dim).transpose(1, 2)

        # RoPE
        past_len = kv_cache[0].shape[1] if kv_cache is not None else 0
        cos, sin = self.rope(past_len + S)
        cos_cur, sin_cur = (
            cos[:, :, past_len : past_len + S, :],
            sin[:, :, past_len : past_len + S, :],
        )
        q_rope, k_rope = apply_rope(q_rope, k_rope, cos_cur, sin_cur)

        # Cache
        if kv_cache is not None:
            c_kv = torch.cat([kv_cache[0], c_kv], dim=1)
            k_rope = torch.cat([kv_cache[1], k_rope], dim=2)
        new_cache = (c_kv, k_rope) if use_cache else None

        # Decompress → GQA expand
        S_kv = c_kv.shape[1]
        k_nope = (
            self.w_uk(c_kv)
            .view(B, S_kv, self.num_kv_heads, self.qk_nope_dim)
            .transpose(1, 2)
        )
        V = (
            self.w_uv(c_kv)
            .view(B, S_kv, self.num_kv_heads, self.v_head_dim)
            .transpose(1, 2)
        )

        k_nope = k_nope.repeat_interleave(self.kv_group_size, dim=1)
        V = V.repeat_interleave(self.kv_group_size, dim=1)
        k_rope_exp = k_rope.expand(-1, self.num_heads, -1, -1)

        Q = torch.cat([q_nope, q_rope], dim=-1)
        K = torch.cat([k_nope, k_rope_exp], dim=-1)

        mask = torch.ones(S, S_kv, dtype=torch.bool, device=x.device).tril(
            diagonal=S_kv - S
        )
        out = F.scaled_dot_product_attention(Q, K, V, attn_mask=mask)

        out = (
            out.transpose(1, 2)
            .contiguous()
            .view(B, S, self.num_heads * self.v_head_dim)
        )
        return self.fc_out(out), new_cache

    # ──────────────────────────────────────────────────────────────────────
    # Absorbed inference: MQA-like, no decompression
    # ──────────────────────────────────────────────────────────────────────
    def _forward_absorbed(self, x, kv_cache=None, use_cache=False):
        B, S, E = x.shape

        # Q — compressed, then absorbed projection to latent space
        c_q = self.q_norm(self.w_dq(x))
        q_absorbed = F.linear(c_q, self._w_uq_absorbed)
        q_absorbed = q_absorbed.view(B, S, self.num_heads, self.kv_latent_rank)
        q_absorbed = q_absorbed.transpose(1, 2)

        q_rope = (
            self.w_qr(c_q).view(B, S, self.num_heads, self.qk_rope_dim).transpose(1, 2)
        )

        # KV — just compress, don't decompress
        kv_out = self.w_dkv(x)
        c_kv, k_rope = torch.split(
            kv_out, [self.kv_latent_rank, self.qk_rope_dim], dim=-1
        )
        c_kv = self.kv_norm(c_kv)
        k_rope = k_rope.view(B, S, 1, self.qk_rope_dim).transpose(1, 2)

        # RoPE
        past_len = kv_cache[0].shape[1] if kv_cache is not None else 0
        cos, sin = self.rope(past_len + S)
        cos_cur, sin_cur = (
            cos[:, :, past_len : past_len + S, :],
            sin[:, :, past_len : past_len + S, :],
        )
        q_rope, k_rope = apply_rope(q_rope, k_rope, cos_cur, sin_cur)

        # Cache
        if kv_cache is not None:
            c_kv = torch.cat([kv_cache[0], c_kv], dim=1)
            k_rope = torch.cat([kv_cache[1], k_rope], dim=2)
        new_cache = (c_kv, k_rope) if use_cache else None

        S_kv = c_kv.shape[1]

        # Keys: [c_kv ; k_rope]
        k_latent = c_kv.unsqueeze(1).expand(-1, self.num_heads, -1, -1)
        k_rope_exp = k_rope.expand(-1, self.num_heads, -1, -1)

        Q = torch.cat([q_absorbed, q_rope], dim=-1)  # (B, H, S, r_kv + d_rope)
        K = torch.cat([k_latent, k_rope_exp], dim=-1)  # (B, H, S_kv, r_kv + d_rope)

        # Values: pad c_kv with zeros to match Q/K last dim for SDPA compatibility
        # SDPA on some backends (MPS, certain flash paths) requires equal head dims
        v_latent = c_kv.unsqueeze(1).expand(
            -1, self.num_heads, -1, -1
        )  # (B, H, S_kv, r_kv)
        v_pad = torch.zeros(
            B,
            self.num_heads,
            S_kv,
            self.qk_rope_dim,
            dtype=v_latent.dtype,
            device=v_latent.device,
        )
        V = torch.cat([v_latent, v_pad], dim=-1)  # (B, H, S_kv, r_kv + d_rope)

        mask = torch.ones(S, S_kv, dtype=torch.bool, device=x.device).tril(
            diagonal=S_kv - S
        )
        out = F.scaled_dot_product_attention(Q, K, V, attn_mask=mask)
        # out: (B, H, S, r_kv + d_rope) — but last d_rope dims are zero contributions

        # Slice off the zero-padded portion before the output projection
        out = out[..., : self.kv_latent_rank]  # (B, H, S, r_kv)

        out = out.transpose(1, 2)  # (B, S, H, r_kv)
        out = torch.einsum("bshc,hec->bse", out, self._w_o_absorbed)  # (B, S, E)

        return out, new_cache

    # ──────────────────────────────────────────────────────────────────────
    # Dispatch
    # ──────────────────────────────────────────────────────────────────────
    def forward(self, x, kv_cache=None, use_cache=False):
        if self.training:
            return self._forward_train(x, kv_cache, use_cache)
        else:
            if not self._absorbed_ready:
                self._build_absorbed_weights()
            return self._forward_absorbed(x, kv_cache, use_cache)


# ── Norms ─────────────────────────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, embed_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(embed_size))
        self.eps = eps

    def forward(self, x):
        # Cast to float32 for the norm computation — critical for fp16 stability.
        # rsqrt of a near-zero mean^2 in fp16 easily produces inf/NaN.
        x_f32 = x.float()
        norm = x_f32 * torch.rsqrt(x_f32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(x.dtype)


# ── MLP (SwiGLU) ──────────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, embed_size, expansion_factor=4):
        super().__init__()
        hidden = int(embed_size * expansion_factor * 2 / 3)
        hidden = (hidden + 7) // 8 * 8
        self.gate = nn.Linear(embed_size, hidden, bias=False)
        self.up = nn.Linear(embed_size, hidden, bias=False)
        self.down = nn.Linear(hidden, embed_size, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


# ── MoE ───────────────────────────────────────────────────────────────────────
class MixtureOfExperts(nn.Module):
    def __init__(self, embed_size, num_experts=4, top_k=2, expansion_factor=4):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        hidden = int(embed_size * expansion_factor * 2 / 3)
        hidden = (hidden + 7) // 8 * 8

        # Init in float32 regardless of model dtype — moved to device dtype by .to()
        self.w_gate = nn.Parameter(torch.empty(num_experts, embed_size, hidden))
        self.w_up = nn.Parameter(torch.empty(num_experts, embed_size, hidden))
        self.w_down = nn.Parameter(torch.empty(num_experts, hidden, embed_size))
        self.router = nn.Linear(embed_size, num_experts, bias=False)

        for w in (self.w_gate, self.w_up, self.w_down):
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))

    def forward(self, x):
        B, S, E = x.shape
        T = B * S
        x_flat = x.view(T, E)

        logits = self.router(x_flat)
        topk_w, topk_ids = torch.topk(logits, self.top_k, dim=-1)
        topk_w = F.softmax(topk_w, dim=-1)

        gate = torch.zeros(T, self.num_experts, device=x.device, dtype=x.dtype)
        gate.scatter_(1, topk_ids, topk_w)

        h_gate = torch.einsum("te,neh->tnh", x_flat, self.w_gate)
        h_up = torch.einsum("te,neh->tnh", x_flat, self.w_up)
        h = F.silu(h_gate) * h_up
        h = h * gate.unsqueeze(-1)
        out = torch.einsum("tnh,nhe->te", h, self.w_down)

        return out.view(B, S, E)


# ── Block ─────────────────────────────────────────────────────────────────────
class Block(nn.Module):
    def __init__(
        self,
        embed_size,
        num_heads,
        num_kv_heads,
        qk_nope_dim,
        qk_rope_dim,
        kv_latent_rank,
        q_latent_rank,
        v_head_dim,
        max_seq_length,
    ):
        super().__init__()
        self.attn = Attention(
            embed_size,
            num_heads,
            num_kv_heads,
            qk_nope_dim,
            qk_rope_dim,
            kv_latent_rank,
            q_latent_rank,
            v_head_dim,
            max_seq_length,
        )
        self.ffn = MLP(embed_size)
        self.norm1 = RMSNorm(embed_size)
        self.norm2 = RMSNorm(embed_size)

    def forward(self, x, kv_cache=None, use_cache=False):
        attn_out, cache = self.attn(
            self.norm1(x), kv_cache=kv_cache, use_cache=use_cache
        )
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, cache


# ── LLM ───────────────────────────────────────────────────────────────────────
class LLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_size,
        num_blocks,
        num_heads,
        num_kv_heads,
        qk_nope_dim,
        qk_rope_dim,
        kv_latent_rank,
        q_latent_rank,
        v_head_dim,
        max_seq_length,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.blocks = nn.ModuleList(
            [
                Block(
                    embed_size,
                    num_heads,
                    num_kv_heads,
                    qk_nope_dim,
                    qk_rope_dim,
                    kv_latent_rank,
                    q_latent_rank,
                    v_head_dim,
                    max_seq_length,
                )
                for _ in range(num_blocks)
            ]
        )
        self.norm_out = RMSNorm(embed_size)
        self.output_layer = nn.Linear(embed_size, vocab_size, bias=False)
        # Weight tying: safe because both stay in the same dtype after .to()
        self.output_layer.weight = self.embedding.weight

        # Initialise embedding with small std — prevents fp16 overflow at start
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)

    def forward(self, x, past_kv=None, use_cache=False):
        x = self.embedding(x)
        new_kv = []
        for i, block in enumerate(self.blocks):
            layer_cache = past_kv[i] if past_kv is not None else None
            x, cache = block(x, kv_cache=layer_cache, use_cache=use_cache)
            new_kv.append(cache)
        out = self.output_layer(self.norm_out(x))
        return (out, new_kv) if use_cache else out


model = LLM(
    vocab_size=len(tokenizer),
    embed_size=128,
    num_blocks=4,
    num_heads=32,
    num_kv_heads=None,
    qk_nope_dim=None,
    qk_rope_dim=None,
    kv_latent_rank=None,
    q_latent_rank=None,
    v_head_dim=None,
    max_seq_length=512,
).to(device)

summary(model, input_size=(1, 512), dtypes=[torch.long], device=device)


Layer (type:depth-idx)                                  Output Shape              Param #
LLM                                                     [1, 512, 96]              --
├─Embedding: 1-1                                        [1, 512, 128]             12,288
├─ModuleList: 1-2                                       --                        --
│    └─Block: 2-1                                       [1, 512, 128]             --
│    │    └─RMSNorm: 3-1                                [1, 512, 128]             128
│    │    └─Attention: 3-2                              [1, 512, 128]             55,552
│    │    └─RMSNorm: 3-3                                [1, 512, 128]             128
│    │    └─MLP: 3-4                                    [1, 512, 128]             132,096
│    └─Block: 2-2                                       [1, 512, 128]             --
│    │    └─RMSNorm: 3-5                                [1, 512, 128]             128
│    │    └─Attention: 3-6                  

In [6]:
def reward_fn(response: str) -> float:
    special_char = response[0] if response else "+"
    num_special = response.count(special_char) - 1

    return math.pow(num_special, 2)


@torch.no_grad()
def generate(
    model, tokenizer, prompt, n=8, max_length=64, temperature=1.0, verbose=True
):
    was_training = model.training
    model.eval()

    input_ids = (
        torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device)
        .unsqueeze(0)
        .expand(n, -1)
        .contiguous()
    )

    prompt_len = input_ids.shape[1]
    if verbose:
        print(f"Prompt tokens: {input_ids.shape}")

    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()
    start_time = perf_counter_ns()

    all_sampled_lp = []
    past_kv = None

    # Prefill: process entire prompt, populate cache
    with torch.autocast(device_type=device.type, dtype=dtype):
        logits, past_kv = model(input_ids, use_cache=True)

    for _ in range(max_length - prompt_len):
        next_token_logits = logits[:, -1, :].float() / temperature
        log_probs = F.log_softmax(next_token_logits, dim=-1)
        probs = log_probs.exp()
        next_token = torch.multinomial(probs, num_samples=1)

        all_sampled_lp.append(log_probs.gather(1, next_token))
        input_ids = torch.cat([input_ids, next_token], dim=1)

        # Decode: only feed the new token, reuse cached K/V
        with torch.autocast(device_type=device.type, dtype=dtype):
            logits, past_kv = model(next_token, past_kv=past_kv, use_cache=True)

    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()
    end_time = perf_counter_ns()

    if was_training:
        model.train()

    log_probs_tensor = torch.cat(all_sampled_lp, dim=1)
    decoded = tokenizer.decode_batch(input_ids.tolist())

    if verbose:
        n_new = input_ids.size(1) - prompt_len
        print(
            f"Total Time: {(end_time - start_time) / 1e9:.4f}s  |  "
            f"{n_new / (end_time - start_time):.1f} tok/s"
        )

    return decoded, log_probs_tensor


decoded, log_probs = generate(model, tokenizer, max_length=128, n=64, prompt=":")
for response in decoded:
    print(f"Reward: {reward_fn(response)} | Response: {response}")

Prompt tokens: torch.Size([64, 29])
Total Time: 0.5370s  |  0.0 tok/s
Reward: 9.0 | Response: :=968I6>{2t=b0a4Tz-P
{?qM >G_D~T&.6;cJ~}t;?:a`IDz{q8*O4n@V2:
raVETD&r(nYeursW<R`@fk0a@~;oF:to{$3

Reward: 4.0 | Response: :v9_.1xf_(8_P*N>>aRNk^M~==Lh>*RW;u>b?%4vim/o<d?BP098 H[jj17j-hez:AQ
&+isDM2EuuBcXoFVuH N
:t]7ubv
Reward: 1.0 | Response: :45T#&2jL;;RC8<W.9:qPE4(m=3zsd4gSi<XDfD3WiZY)**qJ+->Uj
38jN3j0eG=)@R7Cwm%i<cj8<I-VDNiJMo6F6A}H{L3(
Reward: 4.0 | Response: :`PEir?RVS4OuGKjSa{n%9n3E]
e9]pGih8?82[[yGz:%S?:65?>zf7N`%LQy$~TqrVU`)4@E;+qV~w*Y^{g[tV4
0vatSJ>WVr
Reward: 1.0 | Response: :!*phxq2BhlC8|:`d0nveJo8jj7muayLftv}DJt
PD&ta0;Aq(6(lHZRi|=WLa+*-#VE|U1HkS*74#yh
6rX&?<@J6F_HN
Reward: 1.0 | Response: :(IV3E582OP1tY~7McaT[,(ET,!S[j]u1Ut)?$r2pFePw#
e@fyF}:&-P99$3`[X?~k4C
VgtwR1r&S{!4]H!lN./!vVU|
Reward: 4.0 | Response: :vq#6~{+g8lq|yFT&>
8hqhru)<E:~hq:rY)4a@$N1Idlq$sZ=4~6Lh`4e5lFKZ}@21<9]$q;
2D*yLS}qj3~+M}P0a~RE#}<f
Reward: 0.0 | Response: :emC3KJQG#/>=BFuuLyI[L TVRSBX)&EsS{DyuMv8BPs+}rDA2`=^x+

In [18]:
def get_log_probs_for_tokens(
    model: nn.Module,
    input_ids: torch.Tensor,
    prompt_len: int,
    max_new: int,
) -> torch.Tensor:
    with torch.autocast(device_type=device.type, dtype=dtype):
        logits = model(input_ids[:, :-1])
    logits = logits.float()
    gen_logits = logits[:, prompt_len - 1 : prompt_len - 1 + max_new, :]
    gen_ids = input_ids[:, prompt_len : prompt_len + max_new]
    log_probs = F.log_softmax(gen_logits, dim=-1)
    return log_probs.gather(-1, gen_ids.unsqueeze(-1)).squeeze(-1)


@torch.no_grad()
def generate_grpo(model, input_ids, max_new, temperature):
    B, prompt_len = input_ids.shape

    # Pre-allocate the full sequence buffer
    sequences = torch.empty(
        B, prompt_len + max_new, dtype=torch.long, device=input_ids.device
    )
    sequences[:, :prompt_len] = input_ids

    per_tok_lp = []
    past_kv = None

    # Prefill
    with torch.autocast(device_type=device.type, dtype=dtype):
        logits, past_kv = model(input_ids, use_cache=True)

    for step in range(max_new):
        next_logits = logits[:, -1, :].float() / temperature
        lp = F.log_softmax(next_logits, dim=-1)
        tok = torch.multinomial(lp.exp(), num_samples=1)
        per_tok_lp.append(lp.gather(1, tok))

        sequences[:, prompt_len + step] = tok.squeeze(1)

        with torch.autocast(device_type=device.type, dtype=dtype):
            logits, past_kv = model(tok, past_kv=past_kv, use_cache=True)

    old_lp = torch.cat(per_tok_lp, dim=1)  # this one cat is fine — it's once at the end
    return sequences, old_lp


def compute_grpo_loss(
    policy: nn.Module,
    ref_policy: nn.Module,
    prompts: torch.Tensor,
    tokenizer,
    reward_fn,
    G: int = 4,
    max_new: int = 64,
    temperature: float = 1.0,
    epsilon: float = 0.02,
    beta: float = 0.1,
) -> tuple[torch.Tensor, float, float]:
    B, prompt_len = prompts.shape

    # start_step1 = perf_counter_ns()
    # 1. Sample — no grad, autocast handled inside generate_grpo
    expanded = prompts.repeat_interleave(G, dim=0)
    sequences, old_lp = generate_grpo(policy, expanded, max_new, temperature)
    # end_step1 = perf_counter_ns()

    # start_step2 = perf_counter_ns()
    # 2. Rewards (CPU-bound string ops — no autocast needed)
    gen_part = sequences[:, prompt_len:]
    responses = tokenizer.decode_batch(gen_part.tolist())
    rewards = torch.tensor(
        [reward_fn(r) for r in responses], dtype=torch.float32, device=device
    )
    # end_step2 = perf_counter_ns()

    # start_step3 = perf_counter_ns()
    # 3. Group-normalised advantages (float32, no autocast)
    rewards_g = rewards.view(B, G)
    mean_r = rewards_g.mean(dim=1, keepdim=True)
    std_r = rewards_g.std(dim=1, keepdim=True) + 1e-8
    advantages = ((rewards_g - mean_r) / std_r).view(B * G).detach()
    # end_step3 = perf_counter_ns()

    # start_step4 = perf_counter_ns()
    # 4–5. Log-probs — autocast handled inside get_log_probs_for_tokens
    new_lp = get_log_probs_for_tokens(policy, sequences, prompt_len, max_new)
    with torch.no_grad():
        ref_lp = get_log_probs_for_tokens(ref_policy, sequences, prompt_len, max_new)
    # end_step4 = perf_counter_ns()

    # start_step5 = perf_counter_ns()
    # 6–8. Loss (float32 math)
    new_seq_lp = new_lp.sum(dim=1)
    old_seq_lp = old_lp.sum(dim=1).detach()
    ref_seq_lp = ref_lp.sum(dim=1)

    ratio = torch.exp(new_seq_lp - old_seq_lp)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()

    kl = (new_seq_lp - ref_seq_lp).mean()
    loss = policy_loss + beta * kl
    # end_step5 = perf_counter_ns()

    # Print timing breakdown
    # print(f"Step 1 (sample): {(end_step1 - start_step1) / 1e6:.2f} ms")
    # print(f"Step 2 (reward): {(end_step2 - start_step2) / 1e6:.2f} ms")
    # print(f"Step 3 (advantages): {(end_step3 - start_step3) / 1e6:.2f} ms")
    # print(f"Step 4 (log-probs): {(end_step4 - start_step4) / 1e6:.2f} ms")
    # print(f"Step 5 (loss): {(end_step5 - start_step5) / 1e6:.2f} ms")

    return loss, rewards.mean().item(), kl.item()

In [24]:
import copy
from tqdm import trange

ref_model = copy.deepcopy(model)
for p in ref_model.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.01)

PROMPTS = [":", "!", "#", "@", "?"]
G = 64
MAX_NEW = 64
STEPS = 50
LOG_EVERY = 1

model.train()

pbar = trange(1, STEPS + 1, desc="Training")
for step in pbar:
    idxs = torch.randint(len(PROMPTS), (4,)).tolist()
    batch = torch.tensor(
        tokenizer.encode_batch([PROMPTS[i] for i in idxs]),
        dtype=torch.long,
        device=device,
    )

    # No outer autocast — it's scoped to forward passes inside compute_grpo_loss
    loss, mean_reward, kl = compute_grpo_loss(
        model,
        ref_model,
        batch,
        tokenizer,
        reward_fn,
        G=G,
        max_new=MAX_NEW,
    )

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    pbar.set_postfix(
        {
            "loss": f"{loss.item():.4f}",
            "reward": f"{mean_reward:.3f}",
            "KL": f"{kl:.5f}",
        }
    )


Training:  22%|██▏       | 11/50 [00:04<00:16,  2.38it/s, loss=0.3209, reward=150.484, KL=3.21008]


KeyboardInterrupt: 

In [25]:
print("\n── Sample after training ──")
sequences, log_probs = generate_grpo(model, batch, max_new=MAX_NEW, temperature=1.0)
decoded_responses = tokenizer.decode_batch(sequences.tolist())
for response, lp in zip(decoded_responses, log_probs):
    print(
        f"Reward: {reward_fn(response)} | Response: {response} | Log-prob: {lp.sum().item():.2f}"
    )



── Sample after training ──
Reward: 1.0 | Response: @Cq*qqI~q`CtqqsWqC/odAq=qqyvqqq3qjqqeQ>qhJ9hvAqqqqaqqVJP&pI@quqqq | Log-prob: -206.69
Reward: 1.0 | Response: @7qMqqhqqqZWP+EqqFhUfR(qCqbYq9dCHqpNEPaqqqzTqgVqB)KrVq_7|@TJWqqq | Log-prob: -236.31
Reward: 1.0 | Response: ?q>qqqBq?RG:jqq@P+`i~q~qqQ5<]d7qWqJZ@ C|Kqqq$uh;qqq3>Iy&qmvq*@C]g | Log-prob: -235.23
Reward: 0.0 | Response: ?;qq,qqLaj/qRdqqq!:ab|q:74GqqbqqBUqcqqqwT^qUqe^giqJqqqhqq3qqa{q: | Log-prob: -209.05


In [27]:
batch.shape

torch.Size([4, 29])